# NB12 — PRECISE domain adaptation

Report transfer gain vs no adaptation, and separately from deconvolution (NB02).
No fixed numeric gate.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
N_PC, N_PV = 20, 10
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from transforms import precise


In [ ]:
# Load tumour vs cell-line expression
tumour_p = INTERIM / "intrinsic_expression.parquet"
bulk_p = INTERIM / "harmonised_expression.parquet"
tumour = pd.read_parquet(tumour_p) if tumour_p.exists() else (pd.read_parquet(bulk_p) if bulk_p.exists() else None)
dep = REPO_ROOT / "depmap_data" / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"


In [ ]:
# Compute
gain_precise = gain_deconv = float("nan")
if tumour is not None and dep.exists():
    T = tumour.select_dtypes(include=[np.number])
    T.columns = T.columns.astype(str).str.upper()
    from demo_patients import is_excluded, load_demo_exclude_ids
    _ex = load_demo_exclude_ids(REF / "demo_patients.json")
    if _ex:
        T = T.loc[[i for i in T.index if not is_excluded(str(i), _ex)]]
        print("NB12 PRECISE excluded demo patients; tumour n=", len(T))
    # read a gene subset from DepMap to avoid loading 1.5GB fully if possible
    header = pd.read_csv(dep, nrows=0)
    gene_cols = [c for c in header.columns if c.split(" ")[0].upper() in set(T.columns)]
    use_cols = [header.columns[0]] + gene_cols[: min(400, len(gene_cols))]
    S = pd.read_csv(dep, usecols=lambda c: c in set(use_cols))
    S = S.set_index(S.columns[0])
    S.columns = [c.split(" ")[0].upper() for c in S.columns]
    common = [g for g in T.columns if g in S.columns][:200]
    if len(common) >= 20:
        Xt = T[common].fillna(0).to_numpy()
        Xs = S[common].fillna(0).to_numpy()
        n_pc = min(N_PC, Xt.shape[0]-1, Xs.shape[0]-1, len(common))
        pv_s, pv_t, angles = precise(Xs, Xt, n_pc=n_pc, n_pv=min(N_PV, n_pc))
        np.savez(ARTIFACTS / "precise_projection.npz", pv_source=pv_s, pv_target=pv_t, angles=angles, genes=np.array(common))
        # dummy response = first PC of tumours; compare CV R2 in original vs aligned subspace
        y = Xt[:, 0]
        r2_raw = float(cross_val_score(Ridge(), Xt, y, cv=5, scoring="r2").mean())
        Zt = Xt @ pv_t.T
        r2_al = float(cross_val_score(Ridge(), Zt, y, cv=5, scoring="r2").mean())
        gain_precise = r2_al - r2_raw
        # deconvolution gain: intrinsic vs harmonised bulk if both exist
        if tumour_p.exists() and bulk_p.exists():
            B = pd.read_parquet(bulk_p).select_dtypes(include=[np.number])
            B.columns = B.columns.astype(str).str.upper()
            c2 = [g for g in common if g in B.columns]
            if c2:
                r2_bulk = float(cross_val_score(Ridge(), B[c2].fillna(0), B[c2].fillna(0).to_numpy()[:, 0], cv=5, scoring="r2").mean())
                r2_int = float(cross_val_score(Ridge(), T[c2].fillna(0), T[c2].fillna(0).to_numpy()[:, 0], cv=5, scoring="r2").mean())
                gain_deconv = r2_int - r2_bulk
        print("angles (deg)", np.degrees(angles)[:8])
pd.Series({"gain_precise": gain_precise, "gain_deconv": gain_deconv}).to_json(INTERIM / "NB12_transfer_gain.json")
print("PRECISE gain", gain_precise, "deconv gain", gain_deconv)


In [ ]:
# GATE — report-only
val = 0.0 if pd.isna(gain_precise) else float(gain_precise)
gate("NB12", "precise_transfer_gain_logged", 1.0, 1.0,
     n=None if tumour is None else int(len(tumour)),
     note=f"delta_precise={gain_precise} delta_deconv={gain_deconv} (no fixed threshold)")


In [ ]:
import matplotlib.pyplot as plt
p = ARTIFACTS / "precise_projection.npz"
if p.exists():
    d = np.load(p, allow_pickle=True)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(np.degrees(d["angles"]))
    ax.set_ylabel("angle (deg)"); ax.set_xlabel("principal vector")
    fig.tight_layout(); fig.savefig(FIGURES / "NB12_angles.png", dpi=140)
